# 📈 Market Pulse — Equity Data Pipeline

**Author:** Gideon Tegene  
**Stack:** Python · pandas · yfinance · DuckDB · dbt  
**Architecture:** Medallion (Bronze → Silver → Gold)

---

## Overview

This notebook documents a production-grade financial data pipeline that:

1. **Extracts** daily OHLCV stock data from the yfinance API for 6 equity tickers
2. **Loads** raw data into DuckDB as a Bronze layer (no transformations)
3. **Transforms** data into a clean Silver layer (long format, no nulls)
4. **Models** a Gold layer with financial metrics (moving averages, daily returns)
5. **Orchestrates** Gold layer transformations using dbt SQL models

---

## Architecture Diagram

```
yfinance API
     ↓
[ EXTRACT ]  →  Raw OHLCV data for 6 tickers
     ↓
[ BRONZE ]   →  Raw data stored in DuckDB (no changes)
     ↓
[ SILVER ]   →  Cleaned, long-format, null-free data
     ↓
[ GOLD ]     →  Financial metrics: MA20, MA50, Daily Returns
     ↓
[ dbt ]      →  SQL-based Gold layer models with lineage
```

---

## Tickers Tracked
| Ticker | Company |
|--------|---------|
| AAPL   | Apple Inc. |
| TSM    | Taiwan Semiconductor |
| VOO    | Vanguard S&P 500 ETF |
| AMD    | Advanced Micro Devices |
| WMT    | Walmart Inc. |
| GOOGL  | Alphabet Inc. |

---
## 1. Configuration

Central config for tickers, date range, database path, and data interval.  
All other scripts import from here — single source of truth.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# All pipeline settings defined here. Import into other modules as needed.

# Equity tickers to track
stock_tickers = ['AAPL', 'TSM', 'VOO', 'AMD', 'WMT', 'GOOGL']

# Historical date range
start = "2020-01-01"
end   = "2026-05-01"

# DuckDB database file path
database_file_path = "market_info.db"

# Data granularity — '1d' = daily, '1mo' = monthly
data_interval = "1d"

---
## 2. Extract — Bronze Layer Ingest

Pulls raw OHLCV data from the yfinance API for each ticker.  
Combines all tickers into a single wide-format DataFrame.  
No transformations applied — raw data only.

**Output:** `combined_df` — wide format, MultiIndex columns, ~9,540 rows × 44 columns

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import duckdb

In [ ]:
# ── Extract: Pull OHLCV data from yfinance ─────────────────────────────────────
# Loop through each ticker, download historical data, tag with ticker name,
# append to a list, then concatenate into one combined DataFrame.

df = []  # Empty list to collect per-ticker DataFrames

for ticker in stock_tickers:
    # Download OHLCV data for each ticker
    ohlcv_data = yf.download(ticker, start, end, data_interval)
    
    # Tag each row with its ticker symbol
    ohlcv_data["ticker"] = ticker
    
    # Append to collection list
    df.append(ohlcv_data)

# Concatenate all tickers into one wide DataFrame
combined_df = pd.concat(df)

print(f"Shape: {combined_df.shape}")
combined_df.head()

In [ ]:
# ── Flatten MultiIndex Columns ─────────────────────────────────────────────────
# yfinance returns a MultiIndex column structure e.g. ('Close', 'AAPL')
# Flatten to readable format e.g. 'Close_AAPL'

combined_df.columns = [
    f"{a}_{b}".strip('_') if b else a 
    for a, b in combined_df.columns
]

print("Columns after flattening:")
print(combined_df.columns.tolist())

---
## 3. Load — Write Bronze to DuckDB

Persists raw data into DuckDB as the **Bronze layer**.  
Uses `CREATE OR REPLACE` to make the pipeline idempotent — safe to rerun without duplicating data.

In [ ]:
# ── Connect to DuckDB ──────────────────────────────────────────────────────────
# DuckDB creates the file automatically if it doesn't exist

con = duckdb.connect(database_file_path)
print(f"Connected to: {database_file_path}")

In [ ]:
# ── Write Bronze Layer ─────────────────────────────────────────────────────────
# CREATE OR REPLACE makes this idempotent — running twice won't duplicate data
# DuckDB reads combined_df directly from memory — no intermediate files needed

con.execute("CREATE OR REPLACE TABLE bronze AS SELECT * FROM combined_df")

# Verify write was successful
con.execute("SELECT * FROM bronze LIMIT 5").fetchdf()

---
## 4. Transform — Build Silver Layer

Transforms raw wide-format Bronze data into a clean long-format Silver layer.

**Steps:**
1. Melt wide format → long format (one row per date per metric per ticker)
2. Split `metric_ticker` column into separate `metric` and `ticker` columns
3. Drop junk rows (`ticker` as metric, `Capital Gains`)
4. Drop null values

**Output:** `silver_df` — ~66,780 rows × 4 columns (Date, value, metric, ticker)

In [ ]:
# ── Melt: Wide Format → Long Format ───────────────────────────────────────────
# Wide: one row per date, one column per metric per ticker (e.g. Close_AAPL)
# Long: one row per date per metric per ticker — standard time series format

silver_df = (
    combined_df
    .reset_index()                                          # Bring Date out of index
    .melt(id_vars='Date',                                   # Keep Date as identifier
          var_name='metric_ticker',                         # Column names → rows
          value_name='value')                               # Values → single column
)

print(f"Shape after melt: {silver_df.shape}")
silver_df.head()

In [ ]:
# ── Split metric_ticker into Two Columns ───────────────────────────────────────
# 'Close_AAPL' → metric='Close', ticker='AAPL'
# n=1 ensures we only split on the FIRST underscore
# (handles cases like 'Stock_Splits_AAPL' correctly)

silver_df[['metric', 'ticker']] = (
    silver_df['metric_ticker']
    .str.split('_', n=1, expand=True)
)

# Drop the now-redundant combined column
silver_df = silver_df.drop(columns=['metric_ticker'])

silver_df.head()

In [ ]:
# ── Clean Silver Layer ─────────────────────────────────────────────────────────
# Remove rows where 'ticker' or 'Capital Gains' appear as metrics
# (these are artifacts from the extract step)

silver_df = silver_df[~silver_df['metric'].isin(['ticker', 'Capital Gains'])]

# Drop null values — NaN rows created during the melt from ticker column artifact
silver_df = silver_df.dropna(subset=['value'])

# Confirm clean metric counts — should be ~9,540 per metric
print(silver_df['metric'].value_counts())

In [ ]:
# ── Write Silver Layer to DuckDB ───────────────────────────────────────────────

con.execute("CREATE OR REPLACE TABLE silver AS SELECT * FROM silver_df")

# Verify
con.execute("SELECT * FROM silver LIMIT 5").fetchdf()

---
## 5. Model — Build Gold Layer

Calculates business-ready financial metrics from the clean Silver layer.

**Metrics:**
| Metric | Description |
|--------|-------------|
| `close_price` | Daily closing price |
| `ma_20` | 20-day rolling moving average |
| `ma_50` | 50-day rolling moving average |
| `daily_return` | Day-over-day percentage price change |

**Output:** `gold_df` — 9,540 rows × 5 columns

In [ ]:
# ── Filter Silver for Close Prices Only ───────────────────────────────────────
# All gold layer metrics are derived from closing price

close_df = (
    silver_df
    .loc[silver_df['metric'] == 'Close']       # Filter for Close metric only
    [['Date', 'ticker', 'value']]              # Keep only relevant columns
    .copy()
)

# Sort by ticker and date — required for correct rolling window calculation
close_df = close_df.sort_values(['ticker', 'Date']).reset_index(drop=True)

print(f"Shape: {close_df.shape}")
close_df.head()

In [ ]:
# ── Calculate Financial Metrics ────────────────────────────────────────────────
# transform() keeps results aligned to original DataFrame rows
# rolling(n) creates a sliding window of n periods
# pct_change() calculates day-over-day percentage change

# 20-day moving average — short-term trend signal
close_df['ma_20'] = (
    close_df.groupby('ticker')['value']
    .transform(lambda x: x.rolling(20).mean())
)

# 50-day moving average — medium-term trend signal
close_df['ma_50'] = (
    close_df.groupby('ticker')['value']
    .transform(lambda x: x.rolling(50).mean())
)

# Daily return — percentage price change from previous day
close_df['daily_return'] = (
    close_df.groupby('ticker')['value']
    .transform(lambda x: x.pct_change())
)

# Rename value → close_price for clarity
gold_df = close_df.rename(columns={'value': 'close_price'})

print(f"Shape: {gold_df.shape}")
gold_df.head(25)

In [ ]:
# ── Write Gold Layer to DuckDB ─────────────────────────────────────────────────

con.execute("CREATE OR REPLACE TABLE gold AS SELECT * FROM gold_df")

# Verify all three layers exist
print("Tables in database:")
con.execute("SHOW TABLES").fetchdf()

---
## 6. Analysis — Sample Queries

Query the Gold layer directly from DuckDB.  
This is the serving layer — clean, metric-ready data.

In [ ]:
# ── Average Daily Return by Ticker ────────────────────────────────────────────
# Which ticker had the highest average daily return over the period?

con.execute("""
    SELECT 
        ticker,
        ROUND(AVG(daily_return) * 100, 4) AS avg_daily_return_pct
    FROM gold
    WHERE daily_return IS NOT NULL
    GROUP BY ticker
    ORDER BY avg_daily_return_pct DESC
""").fetchdf()

In [ ]:
# ── Most Recent Close Price and Moving Averages ────────────────────────────────
# Current snapshot — where is each ticker relative to its moving averages?

con.execute("""
    SELECT 
        ticker,
        Date,
        ROUND(close_price, 2) AS close_price,
        ROUND(ma_20, 2)       AS ma_20,
        ROUND(ma_50, 2)       AS ma_50
    FROM gold
    WHERE Date = (SELECT MAX(Date) FROM gold)
    ORDER BY ticker
""").fetchdf()

In [ ]:
# ── Close the DuckDB Connection ────────────────────────────────────────────────
# Always close the connection when done — required for dbt to access the file

con.close()
print("Connection closed.")

---
## 7. dbt — SQL-Based Gold Layer

The Gold layer is also modeled in dbt using window functions.  
dbt handles lineage, testing, and documentation automatically.

**Run from terminal (inside `market_pulse/` folder):**
```bash
dbt run
dbt test
dbt docs generate && dbt docs serve
```

**dbt model (`models/gold_stocks.sql`):**
```sql
{{ config(materialized='table') }}

SELECT
    Date,
    ticker,
    CAST(value AS DOUBLE) AS close_price,
    AVG(CAST(value AS DOUBLE)) OVER (
        PARTITION BY ticker
        ORDER BY Date
        ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
    ) AS ma_20,
    AVG(CAST(value AS DOUBLE)) OVER (
        PARTITION BY ticker
        ORDER BY Date
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS ma_50
FROM {{ source('main', 'silver') }}
WHERE metric = 'Close'
ORDER BY ticker, Date
```

---
## Next Steps

- [ ] Wrap pipeline into a single `pipeline.py` script
- [ ] Add dbt data quality tests (`not_null`, `accepted_values`)
- [ ] Containerize with Docker
- [ ] Add Airflow DAG for scheduled daily runs
- [ ] Deploy to cloud (AWS S3 + Redshift or GCP BigQuery)
- [ ] Add Streamlit dashboard on top of Gold layer